# Hybrid Demucs from Colab

This supports the Demucs source separation model (https://github.com/facebookresearch/demucs/)
This is only for separation with pre-trained models, not training!

You can either upload files manually (slow) or link your Google Drive account.

In [ ]:
!pip install demucs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached dora_search-0.1.12.tar.gz (87 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached julius-0.2.7.tar.gz (59 kB)
  Preparing metadata (setup.py) ... done
  Using cached lameenc-1.8.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl.metadata (9.9 kB)
  Using cached openunmix-1.3.0-py3-none-any.whl.metadata (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 10.2 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=43cba7c970c282cdd9672f80670fe8cbf6f5756c3dcb5d3e72fa2e61ef99c31a
  Stored in directory: /root/.c

In [ ]:
!pip install torchaudio

In [ ]:
!python3 -m pip install -U git+https://github.com/facebookresearch/demucs#egg=demucs

In [ ]:
# Please BE VERY CAREFUL, this will link your entire drive.
# So don't edit code, except the one that says 'Customize the following options',
# or you might mess up your files.
# IF YOU DO NO WANT TO LINK DRIVE, please see below for an alternative!
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model = "htdemucs"
extensions = ["mp3", "wav", "ogg", "flac"]  # we will look for all those file types.
two_stems = "vocals"   # set to 'vocals' to separate into vocal and no-vocal stems

# Options for the output audio.
mp3 = True
mp3_rate = 320
float32 = False  # output as float 32 wavs, unsused if 'mp3' is True.
int24 = False    # output as int24 wavs, unused if 'mp3' is True.
# You cannot set both `float32 = True` and `int24 = True` !!

in_path = '/content/drive/MyDrive/fma_punk_audio'
out_path = '/content/drive/MyDrive/punk_instruments'

In [ ]:
#@title Useful functions, don't forget to execute
import io
from pathlib import Path
import select
from shutil import rmtree
import subprocess as sp
import sys
from typing import Dict, Tuple, Optional, IO

from google.colab import files

def find_files(in_path):
    out = []
    for file in Path(in_path).iterdir():
        if file.suffix.lower().lstrip(".") in extensions:
            out.append(file)
    return out

def copy_process_streams(process: sp.Popen):
    def raw(stream: Optional[IO[bytes]]) -> IO[bytes]:
        assert stream is not None
        if isinstance(stream, io.BufferedIOBase):
            stream = stream.raw
        return stream

    p_stdout, p_stderr = raw(process.stdout), raw(process.stderr)
    stream_by_fd: Dict[int, Tuple[IO[bytes], io.StringIO, IO[str]]] = {
        p_stdout.fileno(): (p_stdout, sys.stdout),
        p_stderr.fileno(): (p_stderr, sys.stderr),
    }
    fds = list(stream_by_fd.keys())

    while fds:
        # `select` syscall will wait until one of the file descriptors has content.
        ready, _, _ = select.select(fds, [], [])
        for fd in ready:
            p_stream, std = stream_by_fd[fd]
            raw_buf = p_stream.read(2 ** 16)
            if not raw_buf:
                fds.remove(fd)
                continue
            buf = raw_buf.decode()
            std.write(buf)
            std.flush()

def separate(inp=None, outp=None):
    inp = inp or in_path
    outp = outp or out_path
    cmd = ["python3", "-m", "demucs.separate", "-o", str(outp), "-n", model]
    if mp3:
        cmd += ["--mp3", f"--mp3-bitrate={mp3_rate}"]
    if float32:
        cmd += ["--float32"]
    if int24:
        cmd += ["--int24"]
    if two_stems is not None:
        cmd += [f"--two-stems={two_stems}"]
    files = [str(f) for f in find_files(inp)]
    if not files:
        print(f"No valid audio files in {in_path}")
        return
    print("Going to separate the files:")
    print('\n'.join(files))
    print("With command: ", " ".join(cmd))
    p = sp.Popen(cmd + files, stdout=sp.PIPE, stderr=sp.PIPE)
    copy_process_streams(p)
    p.wait()
    if p.returncode != 0:
        print("Command failed, something went wrong.")


def from_upload():
    out_path = Path('separated')
    in_path = Path('tmp_in')

    if in_path.exists():
        rmtree(in_path)
    in_path.mkdir()

    if out_path.exists():
        rmtree(out_path)
    out_path.mkdir()

    uploaded = files.upload()
    for name, content in uploaded.items():
        (in_path / name).write_bytes(content)
    separate(in_path, out_path)


In [ ]:
# This can be quite slow, in particular the loading, and saving from GDrive. Please be patient!
# This is from google drive! Also, this will separate all the files inside the MyDrive/demucs folder,
# so when you are happy with the results, remove the songs from there.
separate()

Going to separate the files:
/content/drive/MyDrive/fma_punk_audio/01036_Spray_Paint_Pink_Pus.wav
/content/drive/MyDrive/fma_punk_audio/01037_Spray_Paint_Drive_By_Feeling.wav
/content/drive/MyDrive/fma_punk_audio/01038_Spray_Paint_Spock_Fingers.wav
/content/drive/MyDrive/fma_punk_audio/01039_Spray_Paint_Day_Sniffer.wav
/content/drive/MyDrive/fma_punk_audio/01040_Designer_LOLLIPOP.wav
/content/drive/MyDrive/fma_punk_audio/01041_Designer_TRY_IT_OUT.wav
/content/drive/MyDrive/fma_punk_audio/01042_American_Ice_Age_Vendetta_Kind_Of_Mood.wav
/content/drive/MyDrive/fma_punk_audio/01043_American_Ice_Age_King_Of_Queens.wav
/content/drive/MyDrive/fma_punk_audio/01044_American_Ice_Age_Wolves.wav
/content/drive/MyDrive/fma_punk_audio/01045_American_Ice_Age_Crooked_Numbers.wav
/content/drive/MyDrive/fma_punk_audio/01046_Chastity_Belt_Joke.wav
/content/drive/MyDrive/fma_punk_audio/01047_Chastity_Belt_Seattle_Party.wav
/content/drive/MyDrive/fma_punk_audio/01048_Chastity_Belt_Lydia.wav
/content/drive

100%|██████████| 80.2M/80.2M [00:00<00:00, 198MB/s]


Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /content/drive/MyDrive/punk_instruments/htdemucs
Separating track /content/drive/MyDrive/fma_punk_audio/01036_Spray_Paint_Pink_Pus.wav


100%|██████████████████████████████████████████████| 35.099999999999994/35.099999999999994 [00:02<00:00, 12.07seconds/s]


Separating track /content/drive/MyDrive/fma_punk_audio/01037_Spray_Paint_Drive_By_Feeling.wav


...

100%|██████████████████████████████████████████████| 35.099999999999994/35.099999999999994 [00:01<00:00, 22.83seconds/s]


Separating track /content/drive/MyDrive/fma_punk_audio/00034_Urinals_Black_Hole.wav


100%|██████████████████████████████████████████████| 35.099999999999994/35.099999999999994 [00:01<00:00, 23.05seconds/s]


Separating track /content/drive/MyDrive/fma_punk_audio/00035_Urinals_Sex.wav


100%|██████████████████████████████████████████████| 35.099999999999994/35.099999999999994 [00:01<00:00, 23.08seconds/s]


In [ ]:
# This is manual upload and download :)
from_upload()
!zip -r separated.zip separated
files.download('./separated.zip')